# Task B — Lie Factor, question by question

CDB601220 · Week 3. Work through the cells from top to bottom. Every dataset is loaded from the **data/** folder beside this notebook. All data are synthetic, supplied with the assignment. Charts and written answers stay inside the notebook; no output files are created.

## 1. Import libraries and locate the data folder

In [1]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

DATA = Path('data')
assert DATA.is_dir(), 'Open this notebook inside task3, beside the data folder.'
sns.set_theme(style='whitegrid', palette=['#0072B2', '#D55E00', '#009E73'])
plt.ioff()
BLUE, ORANGE, GREEN = '#0072B2', '#D55E00', '#009E73'
SOURCE = 'Source: synthetic CDB601220 Week 3 assignment data.'

## B1. Load the five published graphics

In [2]:
ink = pd.read_csv(DATA / 'case07_lie_factor.csv')
display(ink)

,chart_id,subject,value_start,value_end,ink_start,ink_end,ink_unit,ink_dimension,how_the_graphic_encodes_it
0,A,Corporate tax rate,30.0,32.0,4.0,24.0,mm,1,"Bars drawn from a 29.6% baseline, so only the ..."
1,B,Federal health budget (Rs bn),40.0,60.0,1.0,1.5,x scale,2,A coin icon scaled 1.5x in BOTH height and width
2,C,Export earnings ($ bn),25.0,30.0,5.0,6.0,cm,1,Bars from zero: 5.0 cm then 6.0 cm
3,D,Air-pollution index,100.0,150.0,8.0,9.0,cm,1,Bars drawn 8.0 cm then 9.0 cm tall
4,E,City population (millions),2.0,8.0,1.0,4.0,x radius,2,A bubble whose RADIUS was set proportional to ...


### Formula
Data effect = (final − initial) / initial. Graphic effect = (final ink / initial ink)^dimension − 1. Lie Factor = graphic effect / data effect. These are the exact calculations in the course toolkit, written directly here so each step is visible.

### Calculate the effect in the data

In [3]:
ink['data_effect'] = (ink.value_end - ink.value_start) / ink.value_start
display(ink[['chart_id', 'value_start', 'value_end', 'data_effect']])

,chart_id,value_start,value_end,data_effect
0,A,30.0,32.0,0.066667
1,B,40.0,60.0,0.500000
2,C,25.0,30.0,0.200000
3,D,100.0,150.0,0.500000
4,E,2.0,8.0,3.000000


### Calculate the effect represented by the ink

In [4]:
ink['graphic_effect'] = (ink.ink_end / ink.ink_start)**ink.ink_dimension - 1
display(ink[['chart_id', 'ink_start', 'ink_end', 'ink_dimension', 'graphic_effect']])

,chart_id,ink_start,ink_end,ink_dimension,graphic_effect
0,A,4.0,24.0,1,5.000
1,B,1.0,1.5,2,1.250
2,C,5.0,6.0,1,0.200
3,D,8.0,9.0,1,0.125
4,E,1.0,4.0,2,15.000


### Calculate the Lie Factor and verdict

In [5]:
ink['lie_factor'] = ink.graphic_effect / ink.data_effect
ink['verdict'] = np.select(
    [ink.lie_factor.between(.95, 1.05), ink.lie_factor > 1.05],
    ['Honest', 'Exaggerates'], default='Understates')
display(ink[['chart_id', 'data_effect', 'graphic_effect', 'lie_factor', 'verdict']])
assert np.allclose(ink.lie_factor, [75, 2.5, 1, .25, 5])

,chart_id,data_effect,graphic_effect,lie_factor,verdict
0,A,0.066667,5.000,75.00,Exaggerates
1,B,0.500000,1.250,2.50,Exaggerates
2,C,0.200000,0.200,1.00,Honest
3,D,0.500000,0.125,0.25,Understates
4,E,3.000000,15.000,5.00,Exaggerates


## B2. Which graphic is honest?

**C alone is honest (LF=1).** A shows only bar tips above a 29.6% baseline (LF=75). B scales both dimensions of a coin, giving a 125% area increase for a 50% budget rise (LF=2.5). D gives a 12.5% length increase for a 50% pollution rise (LF=0.25). E scales radius with population: area rises 1,500% for a 300% population rise (LF=5).

## B3. Which graphics require an area calculation?

In [6]:
area_graphics = ink.loc[ink.ink_dimension == 2].copy()
area_graphics['wrong_lf_if_dimension_1'] = (
    (area_graphics.ink_end / area_graphics.ink_start - 1) / area_graphics.data_effect)
display(area_graphics[['chart_id', 'lie_factor', 'wrong_lf_if_dimension_1']])

,chart_id,lie_factor,wrong_lf_if_dimension_1
1,B,2.5,1.0
4,E,5.0,1.0


**Answer:** B and E require dimension 2. Using dimension 1 fails to square the measured scale/radius and falsely calls both honest.

## B4. Identify the understated graphic

In [7]:
display(ink.loc[ink.lie_factor < .95, ['chart_id', 'subject', 'data_effect', 'graphic_effect', 'lie_factor']])

,chart_id,subject,data_effect,graphic_effect,lie_factor
3,D,Air-pollution index,0.5,0.125,0.25


**Answer:** D hides three quarters of the relative pollution increase. Understatement still distorts the evidence and could benefit an authority or polluter minimizing the apparent deterioration; no actual beneficiary is identified by these synthetic data.

## B5. Audit the dual-axis claim using the actual values

In [8]:
finance = pd.read_csv(DATA / 'case07_dual_axis_series.csv')
finance['profit_pkr_m'] = finance.revenue_pkr_m - finance.cost_pkr_m
display(finance)

,month,revenue_pkr_m,cost_pkr_m,profit_pkr_m
0,Jan,52,44,8
1,Feb,58,46,12
2,Mar,61,47,14
3,Apr,66,49,17
4,May,71,51,20
5,Jun,78,53,25


**Answer:** F has no unique single Lie Factor for its false crossover: the claim compares two quantities under independent affine scales, rather than one specified data effect and its corresponding ink effect. A factor for one chosen series/interval would not audit that false ordering. Actual profits are **8, 12, 14, 17, 20 and 25 PKR million**: revenue exceeds cost throughout. Both series share the same unit, so use one axis.